# Segmentation Analysis

Checks whether a trained model works equally well for different groups of
applicants (by gender, education, age band, occupation, …), not just
overall. A good overall AUC can hide a group where the model ranks risk
poorly, or where its predicted default rate is consistently too high or too low.

Uses the test-set predictions the training notebook already saved
(`test_predictions.parquet`) — nothing is re-scored and no model is loaded.
The group columns are added from `model_dataset_final.parquet` by `SK_ID_CURR`.

**Sections**
1. Settings
2. Load the predictions and add the group columns
3. Measures per group (with a minimum group size)
4. Run for every grouping
5. Charts of default rate and AUC by group
6. Save

**Before running:** the model must have been trained (it writes
`test_predictions.parquet`, which `Credit_Scoring.ipynb` also reads).

**Outputs** (`data/Model-Training/ModelOutput/<model_folder>/SegmentationAnalysis/`)
- `segment_summary.xlsx` — one sheet per grouping, plus a `Flagged` sheet
- `bad_rate_by_segment/<grouping>.png` — e.g. `ORGANIZATION_TYPE.png`
- `auc_by_segment/<grouping>.png`

Run by the `04_model_pipeline` Airflow DAG after training (task
`run_segmentation`), or by hand.

## 1. Imports & Configuration

`MODEL_FOLDER` (which model's predictions to analyse) comes from the
`HOMECREDIT_SCORING_MODEL_FOLDER` environment variable — the same one
`Credit_Scoring.ipynb` uses, so one setting points both notebooks at the
same model. The Airflow DAG sets it; by hand it defaults to `LightGBM_v4`.

`MIN_SEGMENT_SIZE`: a group with very few test applicants (some
`ORGANIZATION_TYPE` values have well under 100) gives an AUC so noisy it
can look like a real problem when it's just chance. Groups smaller than
this, or with no defaulters (or no non-defaulters), are marked
`reliable=False` and get no AUC/KS.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy import stats

# -- Paths -------------------------------------------------------------------
DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")

# Which model's test predictions to analyse (set by the Airflow DAG; same
# variable as Credit_Scoring.ipynb)
MODEL_FOLDER = os.environ.get("HOMECREDIT_SCORING_MODEL_FOLDER", "LightGBM_v4")
MODEL_OUTPUT_DIR = os.path.join(DATA_ROOT, "Model-Training", "ModelOutput", MODEL_FOLDER)
PATH_TEST_PREDICTIONS = os.path.join(MODEL_OUTPUT_DIR, "test_predictions.parquet")
PATH_MODEL_DATASET = os.path.join(DATA_ROOT, "Model-Training", "ModelDataset", "model_dataset_final.parquet")

SEGMENT_OUTPUT_DIR = os.path.join(MODEL_OUTPUT_DIR, "SegmentationAnalysis")
os.makedirs(SEGMENT_OUTPUT_DIR, exist_ok=True)

# Groups smaller than this get no AUC/KS (too noisy). 30 is a common rule
# of thumb, not a statistically derived cut-off.
MIN_SEGMENT_SIZE = 30

print(f"Model folder      : {MODEL_FOLDER}")
print(f"Test predictions  : {PATH_TEST_PREDICTIONS}")
print(f"Model dataset     : {PATH_MODEL_DATASET}")
print(f"Outputs to        : {SEGMENT_OUTPUT_DIR}")

## 2. Load the Predictions and Add the Group Columns

`test_predictions.parquet` holds `SK_ID_CURR`, `TARGET` (did they default)
and `predicted_proba` (the model's PD) for each test applicant. The group
columns come from `model_dataset_final.parquet`, joined by `SK_ID_CURR`.

Two groupings are made here:

- `AGE_BAND` — age in bands (<25, 25–34, …, 65+).
- `HAS_BUREAU_HISTORY` — whether the applicant has any credit bureau
  records (i.e. `BUREAUCREDIT_DAYS_TO_LAST_CREDIT`, one of the @prod
  model's features, isn't blank). Applicants with no bureau records are
  missing a big part of what the model relies on, so this checks the model
  still works for them — the same situation as a brand-new applicant in
  the real-time API.

In [ ]:
predictions_df = pd.read_parquet(PATH_TEST_PREDICTIONS)
print(f"Loaded {len(predictions_df):,} validation predictions")

# Columns used to form groups
segment_cols = [
    "SK_ID_CURR", "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
    "AGE", "BUREAUCREDIT_DAYS_TO_LAST_CREDIT",
]
segment_attrs = pd.read_parquet(PATH_MODEL_DATASET, columns=segment_cols)

segment_attrs["AGE_BAND"] = pd.cut(
    segment_attrs["AGE"], bins=[0, 25, 35, 45, 55, 65, 120],
    labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"],
)
# Any credit bureau records at all?
segment_attrs["HAS_BUREAU_HISTORY"] = np.where(
    segment_attrs["BUREAUCREDIT_DAYS_TO_LAST_CREDIT"].notna(), "Yes", "No"
)

df = predictions_df.merge(segment_attrs, on="SK_ID_CURR", how="left")
print(f"Joined shape: {df.shape} (should still be {len(predictions_df):,} rows)")
assert len(df) == len(predictions_df), "join changed row count -- SK_ID_CURR mismatch between the two files"
df.head(3)

## 3. Measures per Group

For each group (e.g. each `NAME_EDUCATION_TYPE` value):

- `count` and `volume_pct` — how many test applicants, and what share
- `bad_rate` — the share who actually defaulted
- `mean_pd` — the model's average predicted PD. Should be close to `bad_rate`;
  a big gap means the model's predictions are off for this group (poor calibration)
- `auc` and `ks` — how well the model ranks defaulters above non-defaulters
  within the group (only for reliable groups)

An `ALL` row (the whole test set) is added at the top as the yardstick.

Two rough flags (rules of thumb, not statistical tests), reliable groups only:
- `flag_bad_rate` — default rate above 1.5× or below 0.5× the overall rate
- `flag_auc_drop` — AUC more than 0.05 below the overall AUC

In [ ]:
def segment_metrics(frame: pd.DataFrame, segment_col: str, min_n: int = MIN_SEGMENT_SIZE) -> pd.DataFrame:
    """One row per value of segment_col (largest groups first), with an ALL row on top and the two flags."""
    rows = []
    for name, group in frame.groupby(segment_col, observed=True):
        n = len(group)
        # Big enough, and has both defaulters and non-defaulters
        reliable = n >= min_n and group["TARGET"].nunique() == 2
        auc = roc_auc_score(group["TARGET"], group["predicted_proba"]) if reliable else np.nan
        if reliable:
            # KS = largest gap between the PD distributions of defaulters and non-defaulters
            good_pd = group.loc[group["TARGET"] == 0, "predicted_proba"]
            bad_pd = group.loc[group["TARGET"] == 1, "predicted_proba"]
            ks = stats.ks_2samp(bad_pd, good_pd).statistic
        else:
            ks = np.nan
        rows.append({
            "segment": str(name),
            "count": n,
            "volume_pct": n / len(frame),
            "bad_rate": group["TARGET"].mean(),
            "mean_pd": group["predicted_proba"].mean(),
            "auc": auc,
            "ks": ks,
            "reliable": reliable,
        })

    result = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
    # The whole test set, for comparison
    overall_auc = roc_auc_score(frame["TARGET"], frame["predicted_proba"])
    overall = {
        "segment": "ALL", "count": len(frame), "volume_pct": 1.0,
        "bad_rate": frame["TARGET"].mean(), "mean_pd": frame["predicted_proba"].mean(),
        "auc": overall_auc,
        "ks": stats.ks_2samp(
            frame.loc[frame["TARGET"] == 1, "predicted_proba"],
            frame.loc[frame["TARGET"] == 0, "predicted_proba"],
        ).statistic,
        "reliable": True,
    }
    result = pd.concat([pd.DataFrame([overall]), result], ignore_index=True)

    # Rough flags (rules of thumb, reliable groups only): default rate far
    # from the overall rate, or AUC noticeably worse than overall
    overall_bad_rate = overall["bad_rate"]
    result["flag_bad_rate"] = result["reliable"] & (
        (result["bad_rate"] > overall_bad_rate * 1.5) | (result["bad_rate"] < overall_bad_rate * 0.5)
    ) & (result["segment"] != "ALL")
    result["flag_auc_drop"] = result["reliable"] & (result["auc"] < overall_auc - 0.05) & (result["segment"] != "ALL")
    return result

## 4. Run for Every Grouping

Application-form details (gender, education, family status, housing,
income type, occupation, organisation, age band) plus whether the applicant
has any credit bureau history. Prints each grouping's table, then lists
every flagged group.

In [ ]:
SEGMENT_COLUMNS = [
    "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE",
    "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "AGE_BAND", "HAS_BUREAU_HISTORY",
]

segment_results = {col: segment_metrics(df, col) for col in SEGMENT_COLUMNS}

flagged = []
for col, result in segment_results.items():
    print(f"\n=== {col} ===")
    display_cols = ["segment", "count", "volume_pct", "bad_rate", "mean_pd", "auc", "ks", "reliable"]
    print(result[display_cols].to_string(index=False, float_format="{:.4f}".format))
    for _, row in result[(result["flag_bad_rate"]) | (result["flag_auc_drop"])].iterrows():
        flagged.append({"attribute": col, **row.to_dict()})

flagged_df = pd.DataFrame(flagged)
print(f"\n{len(flagged_df)} segment(s) flagged for a bad rate or AUC deviation from the overall validation set.")
if len(flagged_df):
    print(flagged_df[["attribute", "segment", "count", "bad_rate", "auc", "flag_bad_rate", "flag_auc_drop"]].to_string(index=False))


## 5. Charts of Default Rate and AUC by Group

One horizontal bar chart per grouping, largest groups at the top, with the
overall value as a dashed line. Flagged groups are red. The AUC charts
leave out unreliable groups.

Each chart is saved as its own PNG (`bad_rate_by_segment/<grouping>.png`,
`auc_by_segment/<grouping>.png`), with its height matched to the number of
groups so groupings with many values (`ORGANIZATION_TYPE`) stay readable.

In [ ]:
def plot_metric_by_segment(result: pd.DataFrame, metric: str, title: str, ax, pct: bool = False):
    """
    Bar chart of `metric` per group onto `ax`, with the overall value as a
    dashed line and flagged groups in red. Returns False if there's nothing
    to plot (e.g. no reliable groups for AUC).
    """
    plot_df = result[result["segment"] != "ALL"].sort_values("count", ascending=True)
    if metric == "auc":
        plot_df = plot_df[plot_df["reliable"]]   # AUC only for reliable groups
    if plot_df.empty:
        return False
    overall_value = result.loc[result["segment"] == "ALL", metric].iloc[0]
    # Red for flagged groups, blue otherwise
    colors = ["#C44E52" if f else "#4C72B0" for f in plot_df.get("flag_bad_rate" if metric == "bad_rate" else "flag_auc_drop", False)]
    ax.barh(plot_df["segment"], plot_df[metric], color=colors)
    ax.axvline(overall_value, color="black", linestyle="--", linewidth=1, label=f"Overall: {overall_value:.3f}")
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="y", labelsize=8)
    if pct:
        from matplotlib.ticker import PercentFormatter
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(fontsize=8, loc="lower right")
    return True


# One PNG per grouping and measure; height grows with the number of groups
PLOTS = {
    "bad_rate": ("Bad rate by {col} (red = flagged)", True),
    "auc": ("AUC by {col} (red = flagged, unreliable segments omitted)", False),
}
for metric, (title, pct) in PLOTS.items():
    plot_dir = os.path.join(SEGMENT_OUTPUT_DIR, f"{metric}_by_segment")
    os.makedirs(plot_dir, exist_ok=True)
    for col in SEGMENT_COLUMNS:
        n_bars = (segment_results[col]["segment"] != "ALL").sum()
        fig, ax = plt.subplots(figsize=(9, max(2.5, 0.25 * n_bars + 1)))
        if plot_metric_by_segment(segment_results[col], metric, title.format(col=col), ax, pct=pct):
            path = os.path.join(plot_dir, f"{col}.png")
            fig.savefig(path, dpi=150, bbox_inches="tight")
            plt.show()
            print(f"Saved -> {path}")
        else:
            print(f"Skipped {metric} for {col}: no reliable segments to plot")
        plt.close(fig)

## 6. Save

One Excel workbook: a sheet per grouping, plus a `Flagged` sheet listing
every flagged group — worth reviewing before promoting this model to @prod.

In [ ]:
summary_path = os.path.join(SEGMENT_OUTPUT_DIR, "segment_summary.xlsx")
with pd.ExcelWriter(summary_path) as writer:
    for col, result in segment_results.items():
        result.to_excel(writer, sheet_name=col[:31], index=False)
    (flagged_df if len(flagged_df) else pd.DataFrame(columns=["attribute", "segment"])).to_excel(
        writer, sheet_name="Flagged", index=False
    )

print(f"Saved -> {summary_path}")
print("\nDone. Review the 'Flagged' sheet (or the red bars above) before promoting this model to @prod.")
